# Notebook 02 — Mathematical Experiments
## Convergence Analysis of Gradient Descent for Logistic Regression

**Purpose:** Run the four controlled experiments from the blueprint and
analyse convergence behaviour.

**Experiments:**
- A: Learning-rate sensitivity
- B: Feature scaling effect
- C: L2 regularisation effect
- D: Independent implementation check (sklearn comparison)


In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.data_preparation import load_raw_data, prepare_data
from src.logistic_regression import (
    binary_cross_entropy, gradient, gradient_norm,
    hessian, hessian_eigenvalues, condition_number,
)
from src.gradient_descent import batch_gradient_descent, run_experiment_grid
from src.evaluation import optimisation_summary
from src.visualization import (
    plot_loss_curves_comparison, plot_grad_norms_comparison,
    plot_eigenvalue_spectrum, plot_scaling_comparison,
    plot_regularisation_comparison,
)

SEED = 42
np.random.seed(SEED)
%matplotlib inline


---
## Experiment A: Learning-Rate Sensitivity

**Setup:** Scaled features, no regularisation, zero initialisation.
Vary η ∈ {0.001, 0.01, 0.1, 0.5, 1.0, 2.0, 2.5, 3.0}, chosen to sit on both
sides of the step-size ceiling 2/L. All other settings fixed.


In [ ]:
# Load and prepare data (scaled)
df = load_raw_data("../data/raw")
data = prepare_data(df, test_size=0.20, random_state=SEED, scale=True)

X_train = data["X_train"]
y_train = data["y_train"]
X_test  = data["X_test"]
y_test  = data["y_test"]

print(f"Training set: {X_train.shape[0]} samples, {X_train.shape[1]} features")
print(f"Test set:     {X_test.shape[0]} samples")


In [ ]:
# Run GD for several learning rates
learning_rates = [0.001, 0.01, 0.1, 0.5, 1.0, 2.0, 2.5, 3.0]

lr_results = run_experiment_grid(
    X_train, y_train,
    learning_rates=learning_rates,
    lam=0.0,
    max_iter=20000,
    tol=1e-6,
    verbose=False,
)


In [ ]:
# Summary table
rows = []
for lr, res in lr_results.items():
    s = optimisation_summary(res)
    rows.append({"η": lr, **s})

summary_df = pd.DataFrame(rows)
print(summary_df.to_string(index=False))


In [ ]:
# Loss curves comparison
fig = plot_loss_curves_comparison(
    lr_results,
    title="Experiment A: Loss vs Iteration by Learning Rate",
    save_name="exp_a_loss_curves",
    max_iters_plot=3000,
)
plt.show()


In [ ]:
# Gradient norm comparison
fig = plot_grad_norms_comparison(
    lr_results,
    title="Experiment A: Gradient Norm vs Iteration",
    save_name="exp_a_grad_norms",
    max_iters_plot=3000,
)
plt.show()


### Experiment A — Discussion

- Every rate below the ceiling 2/L ≈ 1.46 decreases the loss at every step.
- The ceiling is sufficient, not necessary: η = 2.0 and 2.5 still converge,
  because the curvature near the solution (0.75) is lower than the global
  bound L (1.37). The local limit is 2/0.75 ≈ 2.66, and η = 3.0 oscillates.
- Doubling η halves the iterations (about 10,128 at η = 1.0, 5,063 at η = 2.0).


---
## Experiment B: Feature Scaling

**Setup:** Compare convergence with and without standardisation, no
regularisation. Each problem is run at its own safe step size η = 1/L, where
L is the largest eigenvalue of the Hessian at the starting point. A shared
learning rate would be unfair: any rate that suits the scaled problem makes
the unscaled one diverge.


In [ ]:
# Unscaled data
data_unscaled = prepare_data(df, test_size=0.20, random_state=SEED, scale=False)


def lipschitz_constant(X):
    """Largest eigenvalue of the Hessian at w = 0 (the global curvature bound)."""
    Xa = np.hstack([X, np.ones((X.shape[0], 1))])
    return float(np.linalg.eigvalsh(Xa.T @ Xa / (4 * X.shape[0]))[-1])


L_unscaled = lipschitz_constant(data_unscaled["X_train"])
L_scaled = lipschitz_constant(X_train)
print(f"L unscaled = {L_unscaled:.3e}  ->  step size 1/L = {1 / L_unscaled:.2e}")
print(f"L scaled   = {L_scaled:.4f}  ->  step size 1/L = {1 / L_scaled:.3f}")

print("\n=== UNSCALED ===")
res_unscaled = batch_gradient_descent(
    data_unscaled["X_train"], data_unscaled["y_train"],
    learning_rate=1 / L_unscaled,
    max_iter=20000, tol=1e-6, verbose=True, print_every=5000,
)

print("\n=== SCALED ===")
res_scaled = batch_gradient_descent(
    X_train, y_train,
    learning_rate=1 / L_scaled,
    max_iter=20000, tol=1e-6, verbose=True, print_every=5000,
)


In [ ]:
# Side-by-side plots
fig = plot_scaling_comparison(res_unscaled, res_scaled, save_name="exp_b_scaling")
plt.show()


In [ ]:
# Hessian condition numbers
print(f"Condition number (unscaled): {res_unscaled.condition_number:.2f}")
print(f"Condition number (scaled):   {res_scaled.condition_number:.2f}")

# Eigenvalue spectra
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

if res_unscaled.hessian_eigenvalues is not None:
    axes[0].bar(range(len(res_unscaled.hessian_eigenvalues)),
                np.sort(res_unscaled.hessian_eigenvalues)[::-1])
    axes[0].set_title(f"Unscaled (κ = {res_unscaled.condition_number:.1f})")
    axes[0].set_ylabel("Eigenvalue")

if res_scaled.hessian_eigenvalues is not None:
    axes[1].bar(range(len(res_scaled.hessian_eigenvalues)),
                np.sort(res_scaled.hessian_eigenvalues)[::-1])
    axes[1].set_title(f"Scaled (κ = {res_scaled.condition_number:.1f})")
    axes[1].set_ylabel("Eigenvalue")

fig.suptitle("Hessian Eigenvalue Spectra", fontsize=13)
fig.tight_layout()
fig.savefig("../results/figures/exp_b_eigenvalues.png", dpi=150, bbox_inches="tight")
plt.show()


### Experiment B — Discussion

- Scaling cuts the condition number by roughly seven orders of magnitude.
  (The cell above uses the weight-only Hessian at each run's final point;
  `results/results_summary.md` reports the full-parameter values at the start.)
- The smallest eigenvalue hardly changes; scaling removes a few huge
  eigenvalues created by raw-unit columns such as credit amount.
- The scaled run converges; the unscaled run is still far from the minimum
  after 20,000 iterations because its safe step size is about 2 × 10⁻⁷.


---
## Experiment C: L2 Regularisation

**Setup:** Scaled features, η = 1.0.  Compare λ = 0 vs λ = 1.0.
(`run_all_experiments.py` runs the full grid λ ∈ {0, 0.1, 1, 10, 100}.)


In [ ]:
LAM = 1.0

print("=== Unregularised (λ=0) ===")
res_unreg = batch_gradient_descent(
    X_train, y_train,
    learning_rate=1.0, lam=0.0,
    max_iter=20000, tol=1e-6, verbose=True, print_every=5000,
)

print("\n=== Regularised (λ=1.0) ===")
res_reg = batch_gradient_descent(
    X_train, y_train,
    learning_rate=1.0, lam=LAM,
    max_iter=20000, tol=1e-6, verbose=True, print_every=5000,
)


In [ ]:
fig = plot_regularisation_comparison(res_unreg, res_reg, lam=LAM,
                                    save_name="exp_c_regularisation")
plt.show()

print(f"Condition number (λ=0):   {res_unreg.condition_number:.2f}")
print(f"Condition number (λ={LAM}): {res_reg.condition_number:.2f}")


### Experiment C — Discussion

- The penalty adds λ/m to the Hessian eigenvalues of the weights, which lifts
  the smallest eigenvalue and lowers the condition number.
- Fewer iterations are needed to reach the same gradient tolerance.
- **Note:** the regularised and unregularised objectives are different
  functions — compare convergence behaviour, not raw loss values.


---
## Experiment D: Independent Implementation Check

Compare our GD solution against scikit-learn LogisticRegression.


In [ ]:
from src.evaluation import compare_with_sklearn

# Use a converged result (scaled, η=1.0, λ=0)
best_res = lr_results[1.0]

comparison = compare_with_sklearn(
    X_train, y_train, X_test, y_test,
    w_gd=best_res.w, b_gd=best_res.b,
    lam=0.0,
)

print("=== Implementation Comparison ===")
print(f"sklearn test accuracy:      {comparison['sklearn_accuracy']:.4f}")
print(f"Max |w_gd - w_sk|:          {comparison['max_w_diff']:.6f}")
print(f"Mean |w_gd - w_sk|:         {comparison['mean_w_diff']:.6f}")
print(f"Max |P(y=1)_gd - P(y=1)_sk|: {comparison['max_prob_diff']:.6f}")
print(f"Mean |P(y=1)_gd - P(y=1)_sk|:{comparison['mean_prob_diff']:.6f}")


In [ ]:
# Coefficient comparison scatter
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(comparison["sklearn_w"], best_res.w, alpha=0.6, s=30)
lims = [min(comparison["sklearn_w"].min(), best_res.w.min()) - 0.1,
        max(comparison["sklearn_w"].max(), best_res.w.max()) + 0.1]
ax.plot(lims, lims, "r--", linewidth=1, label="y = x")
ax.set_xlabel("sklearn coefficients")
ax.set_ylabel("Our GD coefficients")
ax.set_title("Coefficient Comparison: GD vs sklearn")
ax.legend()
ax.set_aspect("equal")
fig.tight_layout()
fig.savefig("../results/figures/exp_d_coeff_comparison.png", dpi=150, bbox_inches="tight")
plt.show()


### Experiment D — Discussion

- Coefficients and predicted probabilities agree with scikit-learn to about
  three to five decimal places at this tolerance (tighter with a stricter one).
- This is a sanity check on the implementation, not a correctness proof.


---
## Summary of Mathematical Experiments

| Experiment | Key Finding |
|------------|-------------|
| A: Learning rate | Rates below 2/L are always stable; iterations fall in proportion to 1/η; η = 3.0 oscillates |
| B: Feature scaling | Condition number falls by about seven orders of magnitude; only the scaled run converges |
| C: Regularisation | Larger λ raises the smallest eigenvalue and cuts iterations |
| D: sklearn check | Solutions agree closely; to about 10⁻⁶ with a tight tolerance |

Exact figures are in `results/results_summary.md`, produced by `run_all_experiments.py`.
